# Field dependence over the full zone -- Figures 7 and 8

$\eta^{x;xx}(\hbar\omega, B)$ over the whole Brillouin zone for the TNN model (GGA) with all three field terms, and which term does what (Sec. III B).

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper settings, with a quick setting in the comment. The cached results in `data/` have the resolution in their file names, so changing a line here starts a fresh run for that section.

In [ ]:
N = 900            # full-BZ square grid, same point density as the paper's 600 reciprocal grid; 300 for a quick look
n_omega = 900      # photon energies in the maps; 300 for a quick look
dB_map = 1         # field step (T) in the maps; 10 for a quick look
R_disk = 0.9       # disk radius (1/a) for the band-edge part of the decomposition
h_dec = 0.002      # disk spacing (1/a) for the decomposition; 0.005 for a quick look (0.01 gives wrong signs)

## The field terms at the valleys

MoS$_2$ (TNN) at $B = 30$ T, at the band edge of each valley (spin up at $K$, spin down at $K'$): the moments of the two bands, how much the energy shift moves the gap, and the phase-space factor of the occupied band.

In [ ]:
B = 30.0
p = params("MoS2")
for valley, (k_x, k_y, mask), block in zip(("K", "K'"), valley_grids(0.02, 0.01), (0, 1)):
    k_xv, k_yv = np.meshgrid(k_x, k_y)
    H = zeeman(hamiltonian("TNN", p, k_xv, k_yv), lambda_SOC["MoS2"], B)[block]
    E, V, v_kx, v_ky = bands(H, 0.01)
    Omega, M = berry_and_moment(E, v_kx, v_ky, p["a"])
    i = len(k_x) // 2
    E_res = energy_shift(E, M, B)
    dgap = (E_res[i, i, 1] - E_res[i, i, 0]) - (E[i, i, 1] - E[i, i, 0])
    D_v = phase_space(Omega[i, i, 0], B, p["a"])
    print(f"At {valley}: M_v = {M[i, i, 0]:.3f} mu_B, M_c = {M[i, i, 1]:.3f} mu_B, the energy shift moves the gap by "
          f"{1e3 * dgap:+.3f} meV, and D_v = {D_v:.4f}.")

## Maps $\eta^{x;xx}(\hbar\omega, B)$

$\nu = 10$ meV, $\hbar\omega$ from $0.95E_g$ to $1.05E_{\max}$. Only $B \ge 0$ is computed, since time reversal gives $\eta(-B) = -\eta(B)$. At paper settings this takes 20 to 35 s per field, so 20 to 35 min per material and about 2 h for the four.

In [ ]:
B_map = np.arange(0, 60 + 1e-9, dB_map)
fmaps = {}
for m in materials:
    fn = f"data/map_{m}_N{N}_n{n_omega}_dB{dB_map}.npz"
    if not os.path.exists(fn):
        omega = np.linspace(0.95 * gap(m), 1.05 * E_max(m), n_omega)
        eta = []
        for B in B_map:
            eta.append(eta_mag(m, "TNN", B, omega, 0.01, bz_grid(N))[:, 0, 0, 0])
        np.savez(fn, B=B_map, omega=omega, eta=eta_SI(params(m)["a"]) * np.array(eta))
    fmaps[m] = dict(np.load(fn))
    print(f"{m}: largest |eta(B = 0)| = {np.abs(fmaps[m]['eta'][0]).max():.1e}, largest |eta| = {np.abs(fmaps[m]['eta']).max():.3f} A m V^-2 s^-1")

## Figure 7 -- $\eta^{x;xx}(B)$ at the van Hove energy

The van Hove energy is taken as the photon energy where $|\eta|$ is largest at 60 T. The peak is flat, so $\chi$ at the van Hove energy moves by about 10% between neighbouring points of the $\omega$ grid. The slope is the least-squares fit $\eta = sB$ over $|B| \le 20$ T.

In [ ]:
fig, ax = plt.subplots(figsize=(CW, 2.45))
vH = {}
for m in materials:
    d = fmaps[m]
    j = np.argmax(np.abs(d["eta"][-1]))
    vH[m] = d["omega"][j]
    B_f, eta_f = mirror(d["B"], d["eta"][:, j])
    ax.plot(B_f, eta_f, color=colors[m], label=names[m] + r" ($%.2f$ eV)" % vH[m])
    k = np.abs(B_f) <= 20
    s20 = np.sum(B_f[k] * eta_f[k]) / np.sum(B_f[k]**2)
    print(f"{m}: vH = {vH[m]:.3f} eV, slope over |B| <= 20 T = {s20:.4f} A m V^-2 s^-1 T^-1, eta(60 T) = {eta_f[-1]:.3f}")
zero(ax, v=True)
ax.set_xlabel(r"$B$ (T)")
ax.set_ylabel(r"$\eta^{x;xx}$ ($%s$)" % UNIT)
ax.set_xlim(-62, 62)
ax.legend(loc="upper left")
save(fig, "lpge_field_merged")
plt.show()

## Figure 8 -- maps

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(TW, 1.95), gridspec_kw=dict(wspace=0.62))
for i, m in enumerate(materials):
    d = fmaps[m]
    B_f, eta_f = mirror(d["B"], d["eta"])
    vm = np.abs(eta_f).max()
    im = ax[i].pcolormesh(d["omega"], B_f, eta_f, cmap="RdBu_r", vmin=-vm, vmax=vm, shading="nearest", rasterized=True)
    cb = fig.colorbar(im, ax=ax[i], pad=0.03, fraction=0.07, aspect=22)
    cb.locator = MaxNLocator(6, symmetric=True)
    cb.update_ticks()
    cb.ax.tick_params(labelsize=6, width=0.5, length=2, direction="out")
    cb.outline.set_linewidth(0.5)
    cb.ax.minorticks_off()
    if i == 3:
        cb.set_label(r"$\eta^{x;xx}$ ($%s$)" % UNIT, fontsize=7, labelpad=2)
    ax[i].set_xlabel(r"$\hbar\omega$ (eV)")
    ax[i].set_title(names[m], pad=3, loc="right")
    label(ax[i], "abcd"[i])
    ax[i].tick_params(top=False, right=False, which="both")
    ax[i].xaxis.set_major_locator(MaxNLocator(4))
    ax[i].set_ylim(-60, 60)
    ax[i].yaxis.set_major_locator(MaxNLocator(5, steps=[1, 2, 3, 5, 6, 10]))
ax[0].set_ylabel(r"$B$ (T)")
save(fig, "lpge_map_grid")
plt.show()

## Which field term does what (Sec. III B)

$\chi = \eta^{x;xx}(10\,{\rm T})/10$ T with the field terms switched on one at a time on top of the Zeeman Hamiltonian: Z = Zeeman only, D = phase-space factor, E = energy shift, V = velocity shift, F = all three. At the van Hove energy on the full zone ($\nu = 10$ meV) and at $E_g + 0.1$ eV on the disks ($\nu = 5$ meV). About 2 min per material at paper settings.

In [ ]:
variants = {"Z": "", "D": "D", "E": "E", "V": "V", "F": "EVD"}
decomp = {}
for m in materials:
    fn = f"data/decomp_{m}_N{N}_n{n_omega}_dB{dB_map}_R{R_disk}_h{h_dec}.npz"
    if not os.path.exists(fn):
        si = eta_SI(params(m)["a"])
        out = {}
        for v, terms in variants.items():
            eta_vH = eta_mag(m, "TNN", 10.0, np.array([vH[m]]), 0.01, bz_grid(N), terms=terms)
            eta_edge = eta_mag(m, "TNN", 10.0, np.array([gap(m) + 0.1]), 0.005, valley_grids(R_disk, h_dec), terms=terms)
            out["vH_" + v] = si * eta_vH[0, 0, 0, 0] / 10
            out["edge_" + v] = si * eta_edge[0, 0, 0, 0] / 10
        np.savez(fn, **out)
    decomp[m] = dict(np.load(fn))

print("chi in 1e-2 A m V^-2 s^-1 T^-1 (van Hove) and 1e-4 (band edge)")
print("          " + "".join(f"{v:>9s}" for v in variants) + "     F/Z")
for m in materials:
    d = decomp[m]
    print(f"{m:6s} vH  " + "".join(f"{1e2 * d['vH_' + v]:+9.3f}" for v in variants) + f"  {d['vH_F'] / d['vH_Z']:6.2f}")
    print(f"{m:6s} edge" + "".join(f"{1e4 * d['edge_' + v]:+9.3f}" for v in variants) + f"  {d['edge_F'] / d['edge_Z']:6.2f}")